# CIFAKE — Full Pipeline (Sections 0–4)

Ενιαίο notebook: **Real vs AI-Generated image detection** στο CIFAKE (binary, 32×32, 60k+60k).

| Section | Περιεχόμενο |
|---|---|
| **0** | EDA, stratified split, augmentation, feature extraction (HOG/LBP/Color/ResNet18) |
| **1** | Κλασικοί αλγόριθμοι (k-NN, NB, LogReg, SVM, Decision Tree) |
| **2** | MLP baseline + CNN from scratch (PyTorch) |
| **3** | Transfer learning (full vs frozen) + few-shot + prototypes |
| **4** | SimCLR (self-supervised) + LoRA fine-tuning |

**Εκτέλεση:** top→bottom σε Colab Pro (A100). Τα βαριά artifacts (features, SimCLR encoder) σώζονται στο `features/` με guards, ώστε αν αποσυνδεθεί η session να συνεχίζεις χωρίς recompute.

# Part A — Shared Setup
Τρέξε ΟΛΑ τα κελιά αυτής της ενότητας πρώτα (imports, data, split, helpers).

In [ ]:
!pip install -q scikit-image umap-learn opencv-python-headless kagglehub peft transformers

In [ ]:
import os, glob, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import pandas as pd

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms, models

from skimage.feature import hog, local_binary_pattern
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import umap
warnings.filterwarnings("ignore")

SEED=42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device="cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ---- Config ----
SUBSET_PER_CLASS=5000   # με A100 μπορείς να βάλεις π.χ. 10000 ή None (όλα)
IMG_SIZE=224
HC_SIZE=64
VIZ_SUBSET=3000
BATCH=128
OUT_DIR="features"; os.makedirs(OUT_DIR, exist_ok=True)
IMEAN=[0.485,0.456,0.406]; ISTD=[0.229,0.224,0.225]
CIFAR_MEAN=[0.491,0.482,0.447]; CIFAR_STD=[0.247,0.243,0.262]
CLASSES=["FAKE","REAL"]; n_classes=2

In [ ]:
# Download CIFAKE
import kagglehub
DATA_ROOT=kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print("DATA_ROOT:", DATA_ROOT)
# Εναλλακτικά: DATA_ROOT="/content/cifake"  (πρέπει να περιέχει REAL/ και FAKE/ φακέλους)

In [ ]:
# Σάρωση REAL/FAKE -> paths, labels (+ subsample)
EXTS=("*.jpg","*.jpeg","*.png")
def scan(cls):
    f=[]
    for e in EXTS: f+=glob.glob(os.path.join(DATA_ROOT,"**",cls,e), recursive=True)
    return sorted(f)

paths=[]; labels=[]
for ci,cls in enumerate(CLASSES):
    f=scan(cls)
    if SUBSET_PER_CLASS and len(f)>SUBSET_PER_CLASS:
        f=list(np.random.RandomState(SEED).choice(f, SUBSET_PER_CLASS, replace=False))
    paths+=f; labels+=[ci]*len(f)
labels=np.array(labels)
order=np.random.RandomState(SEED).permutation(len(paths))
paths=[paths[i] for i in order]; labels=labels[order]
print("Συνολικές εικόνες:", len(paths))

# Stratified 70/15/15
all_idx=np.arange(len(paths))
train_idx,temp_idx=train_test_split(all_idx,test_size=0.30,stratify=labels,random_state=SEED)
val_idx,test_idx=train_test_split(temp_idx,test_size=0.50,stratify=labels[temp_idx],random_state=SEED)
print(f"Train {len(train_idx)} | Val {len(val_idx)} | Test {len(test_idx)}")

np.save(f"{OUT_DIR}/labels.npy", labels)
np.save(f"{OUT_DIR}/paths.npy", np.array(paths))
np.savez(f"{OUT_DIR}/splits.npz", train=train_idx, val=val_idx, test=test_idx)

In [ ]:
# Transforms
tf_eval224=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),
                               transforms.Normalize(IMEAN,ISTD)])
tf_aug224=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(),transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(),transforms.Normalize(IMEAN,ISTD)])
tf_plain32=transforms.Compose([transforms.Resize((32,32)),transforms.ToTensor(),
                               transforms.Normalize(CIFAR_MEAN,CIFAR_STD)])
tf_aug32=transforms.Compose([transforms.Resize((32,32)),transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(),transforms.Normalize(CIFAR_MEAN,CIFAR_STD)])

class ImgDS(Dataset):
    # Διαβάζει εικόνες από global paths/labels με δοσμένο transform
    def __init__(self, indices, tf): self.idx=list(indices); self.tf=tf
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        k=self.idx[i]; return self.tf(Image.open(str(paths[k])).convert("RGB")), int(labels[k])

# Γενικές helpers train/eval/plot (χρησιμοποιούνται σε Sections 2-4)
def evaluate(model, loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for xb,yb in loader:
            out=model(xb.to(device)); ps.append(out.argmax(1).cpu().numpy()); ys.append(yb.numpy())
    y=np.concatenate(ys); p=np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p,average="macro"), y, p

def train_torch(model, tr, va, epochs=15, lr=1e-3, wd=0.0, log=True):
    model.to(device)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.Adam(params, lr=lr, weight_decay=wd); crit=nn.CrossEntropyLoss()
    hist={"tr_loss":[],"va_loss":[],"tr_acc":[],"va_acc":[]}
    for ep in range(epochs):
        model.train(); tl=0; cor=0; n=0
        for xb,yb in tr:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(); out=model(xb); loss=crit(out,yb); loss.backward(); opt.step()
            tl+=loss.item()*xb.size(0); cor+=(out.argmax(1)==yb).sum().item(); n+=xb.size(0)
        va_acc,_,_,_=evaluate(model,va)
        hist["tr_loss"].append(tl/n); hist["tr_acc"].append(cor/n); hist["va_acc"].append(va_acc)
        if log: print(f"  ep{ep+1:02d} tr_loss={tl/n:.3f} tr_acc={cor/n:.3f} va_acc={va_acc:.3f}")
    return hist

def plot_history(hist, title=""):
    fig,ax=plt.subplots(1,2,figsize=(10,3.5))
    ax[0].plot(hist["tr_loss"]); ax[0].set_title(f"Train loss {title}"); ax[0].set_xlabel("epoch")
    ax[1].plot(hist["tr_acc"],label="train"); ax[1].plot(hist["va_acc"],label="val")
    ax[1].set_title(f"Accuracy {title}"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
print("Setup OK.")

# Section 0 — EDA & Feature Extraction

In [ ]:
# Βασικές πληροφορίες + κατανομή κλάσεων
counts=np.bincount(labels, minlength=n_classes)
print("Κλάσεις:", CLASSES, "| counts:", counts.tolist(),
      "| imbalance:", round(counts.max()/counts.min(),3))
with Image.open(paths[0]) as im: print("Διάσταση δείγματος:", im.size, im.mode)
plt.figure(figsize=(4,3)); plt.bar(CLASSES,counts,color=["indianred","steelblue"])
plt.title("Κατανομή κλάσεων"); plt.tight_layout(); plt.show()

In [ ]:
# EDA: δείγματα ανά κλάση + intensity hist + προβληματικές
n_show=8
fig,axes=plt.subplots(n_classes,n_show,figsize=(n_show*1.3,n_classes*1.5))
for ci in range(n_classes):
    idx=np.where(labels==ci)[0][:n_show]
    for j,k in enumerate(idx): axes[ci,j].imshow(Image.open(paths[k]).convert("RGB")); axes[ci,j].axis("off")
    axes[ci,0].set_title(CLASSES[ci],loc="left")
plt.suptitle("Δείγματα ανά κλάση"); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,3.5))
for ci in range(n_classes):
    idx=np.where(labels==ci)[0][:500]; h=np.zeros(256)
    for k in idx: h+=np.histogram(np.array(Image.open(paths[k]).convert("L")),bins=256,range=(0,255))[0]
    plt.plot(h/h.sum(),label=CLASSES[ci],alpha=0.85)
plt.legend(); plt.title("Pixel intensity ανά κλάση"); plt.tight_layout(); plt.show()

In [ ]:
# Feature extraction (HOG, LBP, Color, CNN) -- με disk-cache guard
def load_gray(p,s=HC_SIZE): return cv2.cvtColor(cv2.resize(cv2.imread(p),(s,s)),cv2.COLOR_BGR2GRAY)
def load_bgr(p,s=HC_SIZE):  return cv2.resize(cv2.imread(p),(s,s))

def extract_hog(p): return hog(load_gray(p),orientations=9,pixels_per_cell=(8,8),
                               cells_per_block=(2,2),block_norm="L2-Hys",feature_vector=True)
def extract_lbp(p,P=8,R=1):
    lbp=local_binary_pattern(load_gray(p),P,R,method="uniform"); n=P+2
    return np.histogram(lbp.ravel(),bins=n,range=(0,n),density=True)[0].astype(np.float32)
def extract_color(p,bins=32):
    bgr=load_bgr(p); hsv=cv2.cvtColor(bgr,cv2.COLOR_BGR2HSV); fe=[]
    for img,rng in [(bgr,[(0,256)]*3),(hsv,[(0,180),(0,256),(0,256)])]:
        for ch in range(3):
            hh=cv2.calcHist([img],[ch],None,[bins],rng[ch]); fe.append(cv2.normalize(hh,hh).flatten())
    return np.concatenate(fe).astype(np.float32)

feat={}; times={}
if all(os.path.exists(f"{OUT_DIR}/{n}.npy") for n in ["hog","lbp","color","cnn"]):
    feat={"HOG":np.load(f"{OUT_DIR}/hog.npy"),"LBP":np.load(f"{OUT_DIR}/lbp.npy"),
          "Color":np.load(f"{OUT_DIR}/color.npy"),"CNN":np.load(f"{OUT_DIR}/cnn.npy")}
    print("Loaded cached features.")
else:
    for nm,fn,fpath in [("HOG",extract_hog,"hog"),("LBP",extract_lbp,"lbp"),("Color",extract_color,"color")]:
        t0=time.time(); feat[nm]=np.array([fn(p) for p in paths],dtype=np.float32); times[nm]=time.time()-t0
        np.save(f"{OUT_DIR}/{fpath}.npy", feat[nm]); print(f"{nm} {feat[nm].shape} {times[nm]:.1f}s")
    # CNN (ResNet18 frozen 512-d)
    rn=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1); rn.fc=nn.Identity()
    rn.eval().to(device)
    for p in rn.parameters(): p.requires_grad=False
    dl=DataLoader(ImgDS(np.arange(len(paths)),tf_eval224),batch_size=BATCH,shuffle=False,num_workers=2)
    out=[]; t0=time.time()
    with torch.no_grad():
        for xb,_ in dl: out.append(rn(xb.to(device)).cpu().numpy())
    feat["CNN"]=np.concatenate(out).astype(np.float32); times["CNN"]=time.time()-t0
    np.save(f"{OUT_DIR}/cnn.npy", feat["CNN"]); print(f"CNN {feat['CNN'].shape} {times['CNN']:.1f}s")
    del rn; torch.cuda.empty_cache()

In [ ]:
# 2D visualization PCA / t-SNE / UMAP
viz_idx=np.random.RandomState(SEED).choice(len(paths),min(VIZ_SUBSET,len(paths)),replace=False)
y_viz=labels[viz_idx]
def project(X,method):
    Xs=StandardScaler().fit_transform(X[viz_idx])
    if method=="PCA":   return PCA(2,random_state=SEED).fit_transform(Xs)
    if method=="t-SNE": return TSNE(2,random_state=SEED,init="pca",perplexity=30).fit_transform(Xs)
    if method=="UMAP":  return umap.UMAP(n_components=2,random_state=SEED).fit_transform(Xs)
methods=["PCA","t-SNE","UMAP"]
fig,axes=plt.subplots(len(feat),len(methods),figsize=(len(methods)*4,len(feat)*4))
for r,(fn,X) in enumerate(feat.items()):
    for c,m in enumerate(methods):
        emb=project(X,m); ax=axes[r,c]
        for ci,col in zip(range(n_classes),["indianred","steelblue"]):
            mk=y_viz==ci; ax.scatter(emb[mk,0],emb[mk,1],s=6,alpha=0.5,c=col,label=CLASSES[ci])
        ax.set_title(f"{fn} — {m}",fontsize=10); ax.set_xticks([]); ax.set_yticks([])
        if r==0 and c==0: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

dims=pd.DataFrame({"Feature":list(feat.keys()),
                   "Dimensionality":[X.shape[1] for X in feat.values()]})
print(dims.to_string(index=False))

**Q0 notes.** CIFAKE: binary, ισορροπημένο (50/50) → χωρίς ανάγκη χειρισμού imbalance· δυσκολία = χαμηλή ανάλυση + λεπτά generative artifacts. Handcrafted → StandardScaler (fit στο train)· CNN features → προαιρετικά L2-norm. Τα learned CNN features είναι πιο διακριτικά λόγω ιεραρχικών αναπαραστάσεων & inductive bias.

# Section 1 — Κλασικοί Αλγόριθμοι

In [ ]:
MAX_TRAIN=None  # π.χ. 4000 για ταχύτερο CV
def make_splits(X):
    sc=StandardScaler().fit(X[train_idx])
    Xtr=sc.transform(X[train_idx]); ytr=labels[train_idx]
    Xva=sc.transform(X[val_idx]);   yva=labels[val_idx]
    Xte=sc.transform(X[test_idx]);  yte=labels[test_idx]
    if MAX_TRAIN and len(Xtr)>MAX_TRAIN:
        sel=np.random.RandomState(SEED).choice(len(Xtr),MAX_TRAIN,replace=False); Xtr,ytr=Xtr[sel],ytr[sel]
    return Xtr,ytr,Xva,yva,Xte,yte
data={n:make_splits(X) for n,X in feat.items()}

def grid_models():
    return {
      "k-NN":(KNeighborsClassifier(algorithm="brute"),{"n_neighbors":[1,3,5,11,21],"metric":["euclidean","cosine"]}),
      "Logistic Reg":(LogisticRegression(solver="liblinear",max_iter=1000),{"C":[0.01,0.1,1,10,100],"penalty":["l1","l2"]}),
      "SVM":(SVC(),{"kernel":["linear","rbf","poly"],"C":[0.1,1,10,100]}),
      "Decision Tree":(DecisionTreeClassifier(random_state=SEED),{"max_depth":[3,5,10,None],"min_samples_leaf":[1,5,10]}),
    }
def run_grid(est,grid,Xtr,ytr,Xva,yva,Xte,yte):
    gs=GridSearchCV(est,grid,cv=5,scoring="accuracy",n_jobs=-1).fit(Xtr,ytr)
    b=gs.best_estimator_; pr=b.predict(Xte)
    return gs.best_params_,accuracy_score(yva,b.predict(Xva)),accuracy_score(yte,pr),f1_score(yte,pr,average="macro"),pr
def run_nb(Xtr,ytr,Xva,yva,Xte,yte):
    best=None;bn=None;bc=-1
    for nm,clf in [("GaussianNB",GaussianNB()),("BernoulliNB",BernoulliNB())]:
        cv=cross_val_score(clf,Xtr,ytr,cv=5).mean()
        if cv>bc: bc,best,bn=cv,clf,nm
    best.fit(Xtr,ytr); pr=best.predict(Xte)
    return {"type":bn},accuracy_score(yva,best.predict(Xva)),accuracy_score(yte,pr),f1_score(yte,pr,average="macro"),pr

In [ ]:
results=[]; preds={}; t0=time.time()
for fn,(Xtr,ytr,Xva,yva,Xte,yte) in data.items():
    d=DummyClassifier(strategy="most_frequent").fit(Xtr,ytr); pr=d.predict(Xte)
    results.append(["Dummy",fn,"{}",accuracy_score(yva,d.predict(Xva)),accuracy_score(yte,pr),f1_score(yte,pr,average="macro")])
    preds[("Dummy",fn)]=pr
    bp,va,ta,f1,pr=run_nb(Xtr,ytr,Xva,yva,Xte,yte)
    results.append(["Naive Bayes",fn,str(bp),va,ta,f1]); preds[("Naive Bayes",fn)]=pr
    for algo,(est,grid) in grid_models().items():
        bp,va,ta,f1,pr=run_grid(est,grid,Xtr,ytr,Xva,yva,Xte,yte)
        results.append([algo,fn,str(bp),va,ta,f1]); preds[(algo,fn)]=pr
        print(f"{fn:6s}|{algo:14s} test={ta:.3f} f1={f1:.3f}")
clf_time=round(time.time()-t0); print("Χρόνος:", clf_time, "s")
df1=pd.DataFrame(results,columns=["Algorithm","Feature","Best HP","Val Acc","Test Acc","F1-macro"]).round(4)
df1.to_csv(f"{OUT_DIR}/section1_results.csv",index=False)
df1.sort_values(["Algorithm","Test Acc"],ascending=[True,False]).reset_index(drop=True)

In [ ]:
# Q1.2 best combo + confusion matrix
best=df1.loc[df1["Test Acc"].idxmax()]; ba,bf=best["Algorithm"],best["Feature"]
print(f"Καλύτερος: {ba}+{bf} (Test={best['Test Acc']}, F1={best['F1-macro']})")
yte=labels[test_idx]; cm=confusion_matrix(yte,preds[(ba,bf)])
fig,ax=plt.subplots(figsize=(4,4)); im=ax.imshow(cm,cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{ba}+{bf}")
for i in range(2):
    for j in range(2): ax.text(j,i,cm[i,j],ha="center",va="center")
plt.tight_layout(); plt.show()

# Task 1.2 error analysis: λάθη κοινά σε όλους
algos=["k-NN","Naive Bayes","Logistic Reg","SVM","Decision Tree"]; masks=[]
for a in algos:
    sub=df1[(df1.Algorithm==a)&(df1.Feature.isin(list(feat.keys())))]
    bff=sub.loc[sub["Test Acc"].idxmax(),"Feature"]; masks.append(preds[(a,bff)]!=yte)
bad=np.where(np.all(masks,axis=0))[0]; print("Κοινά λάθη:", len(bad))
show=bad[:8]
if len(show):
    fig,ax=plt.subplots(1,len(show),figsize=(len(show)*1.6,2)); ax=np.atleast_1d(ax)
    for a,k in zip(ax,show):
        a.imshow(Image.open(str(paths[test_idx[k]])).convert("RGB")); a.axis("off")
        a.set_title(f"T:{CLASSES[yte[k]]}",fontsize=8)
    plt.suptitle("Κοινά λάθη όλων"); plt.tight_layout(); plt.show()

**Q1 notes.** Q1.2: confusion 2×2 (FAKE↔REAL)· τα FAKE→REAL λάθη = πειστικά synthetic, REAL→FAKE = θολά/θορυβώδη πραγματικά. Q1.3: συνήθως CNN features + απλό μοντέλο > handcrafted + ισχυρό μοντέλο. Q1.4: κοινά λάθη = εγγενώς αμφίσημες 32×32 → και features και data· πρότεινε frequency-domain / noise-residual features.

# Section 2 — MLP & CNN from scratch

In [ ]:
def loaders32(train_tf, batch=128):
    tr=DataLoader(ImgDS(train_idx,train_tf),batch_size=batch,shuffle=True,num_workers=2)
    va=DataLoader(ImgDS(val_idx,tf_plain32),batch_size=batch,shuffle=False,num_workers=2)
    te=DataLoader(ImgDS(test_idx,tf_plain32),batch_size=batch,shuffle=False,num_workers=2)
    return tr,va,te

class MLP(nn.Module):
    def __init__(self,in_dim=3*32*32,hidden=(512,256),n_classes=2,dropout=0.3,batchnorm=True):
        super().__init__(); layers=[]; prev=in_dim
        for h in hidden:
            layers.append(nn.Linear(prev,h))
            if batchnorm: layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout>0: layers.append(nn.Dropout(dropout))
            prev=h
        layers.append(nn.Linear(prev,n_classes)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x.view(x.size(0),-1))

# Task 2.1 MLP
tr,va,te=loaders32(tf_plain32)
mlp=MLP(hidden=(512,256),dropout=0.3)
t0=time.time(); h=train_torch(mlp,tr,va,epochs=15,lr=1e-3); mlp_time=time.time()-t0
plot_history(h,"(MLP)"); acc,f1,_,_=evaluate(mlp,te); mlp_test=acc
mlp_params=sum(p.numel() for p in mlp.parameters())
print(f"[MLP] test={acc:.3f} f1={f1:.3f} params={mlp_params:,} time={mlp_time:.0f}s")
del mlp; torch.cuda.empty_cache()

In [ ]:
# (προαιρετικό) MLP πάνω στα CNN features
sc=StandardScaler().fit(feat["CNN"][train_idx])
def fl(idx,sh): 
    return DataLoader(TensorDataset(torch.tensor(sc.transform(feat["CNN"][idx])),torch.tensor(labels[idx])),
                      batch_size=128,shuffle=sh)
mlp_f=MLP(in_dim=feat["CNN"].shape[1],hidden=(256,),dropout=0.3)
hf=train_torch(mlp_f,fl(train_idx,True),fl(val_idx,False),epochs=15,lr=1e-3,log=False)
accf,f1f,_,_=evaluate(mlp_f,fl(test_idx,False)); mlp_cnnfeat_test=accf; print(f"[MLP on CNN features] test={accf:.3f} f1={f1f:.3f}")
del mlp_f; torch.cuda.empty_cache()

In [ ]:
# Task 2.2 CNN from scratch
class SimpleCNN(nn.Module):
    def __init__(self,n_classes=2,channels=(32,64,128),dropout=0.25,batchnorm=True):
        super().__init__(); blocks=[]; inc=3
        for c in channels:
            blocks+=[nn.Conv2d(inc,c,3,padding=1)]
            if batchnorm: blocks+=[nn.BatchNorm2d(c)]
            blocks+=[nn.ReLU(),nn.MaxPool2d(2)]; inc=c
        self.features=nn.Sequential(*blocks); sp=32//(2**len(channels))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Dropout(dropout),
            nn.Linear(inc*sp*sp,128),nn.ReLU(),nn.Dropout(dropout),nn.Linear(128,n_classes))
    def forward(self,x): return self.classifier(self.features(x))

tra,vaa,tea=loaders32(tf_aug32)
cnn=SimpleCNN()
t0=time.time(); h=train_torch(cnn,tra,vaa,epochs=20,lr=1e-3); cnn_time=time.time()-t0
plot_history(h,"(CNN)"); acc,f1,_,_=evaluate(cnn,tea); cnn_test=acc
cnn_params=sum(p.numel() for p in cnn.parameters())
print(f"[CNN] test={acc:.3f} f1={f1:.3f} params={cnn_params:,} time={cnn_time:.0f}s")
del cnn; torch.cuda.empty_cache()

In [ ]:
# Q2.3 ablation: ποια regularization βοηθά
def quick(channels,bn,dp,aug,epochs=10):
    ld=loaders32(tf_aug32 if aug else tf_plain32)
    m=SimpleCNN(channels=channels,dropout=dp,batchnorm=bn)
    train_torch(m,ld[0],ld[1],epochs=epochs,lr=1e-3,log=False)
    a,f,_,_=evaluate(m,ld[2]); del m; torch.cuda.empty_cache(); return a,f
cfgs={"baseline":dict(channels=(32,64,128),bn=False,dp=0.0,aug=False),
      "+BatchNorm":dict(channels=(32,64,128),bn=True,dp=0.0,aug=False),
      "+Dropout0.5":dict(channels=(32,64,128),bn=True,dp=0.5,aug=False),
      "+DataAug":dict(channels=(32,64,128),bn=True,dp=0.25,aug=True)}
rows=[[k,*[round(x,3) for x in quick(**v)]] for k,v in cfgs.items()]
pd.DataFrame(rows,columns=["Config","Test Acc","F1-macro"])

**Q2 notes.** MLP σε raw pixels αγνοεί χωρική δομή → συχνά χάνει από CNN-features classifiers. CNN υπερτερεί λόγω locality / weight sharing / translation invariance — κρίσιμο για τα τοπικά generative artifacts του CIFAKE.

# Section 3 — Transfer Learning & Few-Shot

In [ ]:
def loaders224(train_indices, train_tf=tf_aug224, batch=64):
    tr=DataLoader(ImgDS(train_indices,train_tf),batch_size=batch,shuffle=True,num_workers=2)
    va=DataLoader(ImgDS(val_idx,tf_eval224),batch_size=batch,shuffle=False,num_workers=2)
    te=DataLoader(ImgDS(test_idx,tf_eval224),batch_size=batch,shuffle=False,num_workers=2)
    return tr,va,te

def build_model(name,strategy):
    if name=="resnet18":
        m=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        if strategy=="frozen":
            for p in m.parameters(): p.requires_grad=False
        m.fc=nn.Linear(m.fc.in_features,2)
    elif name=="efficientnet_b0":
        m=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        if strategy=="frozen":
            for p in m.parameters(): p.requires_grad=False
        m.classifier[1]=nn.Linear(m.classifier[1].in_features,2)
    return m
def n_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

tr_full,va_full,te_full=loaders224(train_idx)
t31=[]; res31={}
for name in ["resnet18","efficientnet_b0"]:
    for strat in ["full","frozen"]:
        print(f"== {name} | {strat} ==")
        m=build_model(name,strat); ntr=n_trainable(m)
        ep=5 if strat=="full" else 6; lr=1e-4 if strat=="full" else 1e-3
        t0=time.time(); train_torch(m,tr_full,va_full,epochs=ep,lr=lr,log=False); dt=time.time()-t0
        acc,f1,y,p=evaluate(m,te_full)
        t31.append([name,strat,round(acc,4),round(f1,4),round(dt,1),ntr]); res31[(name,strat)]=acc
        print(f"   test={acc:.3f} f1={f1:.3f} time={dt:.0f}s params={ntr:,}")
        del m; torch.cuda.empty_cache()
df31=pd.DataFrame(t31,columns=["Model","Strategy","Test Acc","F1-macro","Train Time(s)","#Trainable"])
df31.to_csv(f"{OUT_DIR}/section3_transfer.csv",index=False); df31

In [ ]:
# Task 3.2 few-shot με top-2 configs
def kshot(k,seed=SEED):
    rng=np.random.RandomState(seed); s=[]
    for c in range(n_classes):
        pool=train_idx[labels[train_idx]==c]; s+=list(rng.choice(pool,k,replace=False))
    return np.array(s)
top2=df31.sort_values("Test Acc",ascending=False).head(2)[["Model","Strategy"]].values.tolist()
print("Top-2:",top2)
rows=[]; preds5={}
for name,strat in top2:
    row=[f"{name}/{strat}", df31[(df31.Model==name)&(df31.Strategy==strat)]["Test Acc"].values[0]]
    for k in [10,5]:
        trk=DataLoader(ImgDS(kshot(k),tf_aug224),batch_size=16,shuffle=True,num_workers=2)
        m=build_model(name,strat); ep=20 if strat=="full" else 30
        train_torch(m,trk,va_full,epochs=ep,lr=1e-4 if strat=="full" else 1e-3,log=False)
        acc,f1,y,p=evaluate(m,te_full); row.append(round(acc,4))
        if k==5: preds5[f"{name}/{strat}"]=(p,y)
        del m; torch.cuda.empty_cache()
    rows.append(row)
df32=pd.DataFrame(rows,columns=["Model/Strategy","Full Data","10-shot","5-shot"])
df32.to_csv(f"{OUT_DIR}/section3_fewshot.csv",index=False); df32

In [ ]:
# Q3.4 confusion 5-shot + Task 3.3 prototypes
bk=max(preds5,key=lambda kk:(preds5[kk][0]==preds5[kk][1]).mean()); p,y=preds5[bk]
cm=confusion_matrix(y,p); fig,ax=plt.subplots(figsize=(4,4)); ax.imshow(cm,cmap="Oranges")
ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
ax.set_title(f"5-shot CM — {bk}")
for i in range(2):
    for j in range(2): ax.text(j,i,cm[i,j],ha="center",va="center")
plt.tight_layout(); plt.show()

def build_encoder():
    m=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1); m.fc=nn.Identity()
    for p in m.parameters(): p.requires_grad=False
    return m.eval().to(device)
@torch.no_grad()
def embed(enc,indices):
    dl=DataLoader(ImgDS(indices,tf_eval224),batch_size=64,shuffle=False,num_workers=2); E=[];Y=[]
    for xb,yb in dl: E.append(enc(xb.to(device)).cpu().numpy()); Y.append(yb.numpy())
    return np.concatenate(E),np.concatenate(Y)
enc=build_encoder(); E_test,y_test=embed(enc,test_idx)
def proto_eval(k):
    E_s,y_s=embed(enc,kshot(k)); protos=np.stack([E_s[y_s==c].mean(0) for c in range(n_classes)])
    de=((E_test[:,None,:]-protos[None])**2).sum(-1); pe=de.argmin(1)
    En=E_test/np.linalg.norm(E_test,axis=1,keepdims=True); Pn=protos/np.linalg.norm(protos,axis=1,keepdims=True)
    pc=(En@Pn.T).argmax(1)
    return accuracy_score(y_test,pe),accuracy_score(y_test,pc)
df33=pd.DataFrame([[f"{k}-shot",*[round(x,4) for x in proto_eval(k)]] for k in [10,5]],
                  columns=["Regime","Proto-Euclidean","Proto-Cosine"]); df33

**Q3 notes.** full fine-tuning → καλύτερο acc αλλά πολύ μεγαλύτερος χρόνος/params· frozen → ταχύ & data-efficient (λιγότερο overfitting στο few-shot). Prototype classifier (χωρίς εκπαίδευση βαρών) είναι robust στο 5-shot· cosine ≥ euclidean συνήθως.

# Section 4 — Self-Supervised (SimCLR) & LoRA

## Task 4.1 — SimCLR

In [ ]:
# SimCLR dataset: δύο augmented views ανά εικόνα
simclr_aug=transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE,scale=(0.2,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.1)],p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=23,sigma=(0.1,2.0)),
    transforms.ToTensor(), transforms.Normalize(IMEAN,ISTD)])
class TwoView(Dataset):
    def __init__(self,indices): self.idx=list(indices)
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        img=Image.open(str(paths[self.idx[i]])).convert("RGB")
        return simclr_aug(img), simclr_aug(img)

class SimCLRNet(nn.Module):
    def __init__(self,dim=128):
        super().__init__()
        b=models.resnet18(weights=None); self.fdim=b.fc.in_features; b.fc=nn.Identity(); self.encoder=b
        self.proj=nn.Sequential(nn.Linear(self.fdim,self.fdim),nn.ReLU(),nn.Linear(self.fdim,dim))
    def forward(self,x): return self.proj(self.encoder(x))

def nt_xent(z,temp=0.5):
    N2=z.size(0); N=N2//2; z=F.normalize(z,dim=1)
    sim=z@z.t()/temp; sim.masked_fill_(torch.eye(N2,device=z.device).bool(),-9e15)
    targets=(torch.arange(N2,device=z.device)+N)%N2
    return F.cross_entropy(sim,targets)

In [ ]:
# SimCLR pretraining (με checkpoint guard)
SIMCLR_EPOCHS=30; SIMCLR_BATCH=256
ckpt=f"{OUT_DIR}/simclr_encoder.pt"
simclr=SimCLRNet().to(device); simclr_time=float("nan")
if os.path.exists(ckpt):
    simclr.encoder.load_state_dict(torch.load(ckpt,map_location=device)); print("Loaded SimCLR encoder.")
else:
    dl=DataLoader(TwoView(train_idx),batch_size=SIMCLR_BATCH,shuffle=True,num_workers=2,drop_last=True)
    opt=torch.optim.Adam(simclr.parameters(),lr=3e-4,weight_decay=1e-4)
    _t0=time.time()
    for ep in range(SIMCLR_EPOCHS):
        simclr.train(); tot=0; nb=0
        for v1,v2 in dl:
            x=torch.cat([v1,v2],0).to(device)
            z=simclr(x); loss=nt_xent(z)
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item(); nb+=1
        print(f"ep{ep+1:02d}/{SIMCLR_EPOCHS} NT-Xent={tot/nb:.3f}")
    simclr_time=time.time()-_t0
    torch.save(simclr.encoder.state_dict(),ckpt); print("Saved SimCLR encoder.")

In [ ]:
# Evaluate SimCLR: (1) linear eval, (2) fine-tune full, (3) few-shot fine-tune
class EncoderClf(nn.Module):
    def __init__(self,encoder,freeze):
        super().__init__(); self.encoder=encoder; self.fc=nn.Linear(512,2)
        if freeze:
            for p in self.encoder.parameters(): p.requires_grad=False
    def forward(self,x): return self.fc(self.encoder(x))

@torch.no_grad()
def enc_embed(encoder,indices):
    encoder.eval(); dl=DataLoader(ImgDS(indices,tf_eval224),batch_size=128,shuffle=False,num_workers=2); E=[];Y=[]
    for xb,yb in dl: E.append(encoder(xb.to(device)).cpu().numpy()); Y.append(yb.numpy())
    return np.concatenate(E),np.concatenate(Y)

# (1) Linear eval (logistic regression πάνω σε frozen embeddings)
Etr,ytr=enc_embed(simclr.encoder,train_idx); Ete,yte=enc_embed(simclr.encoder,test_idx)
scs=StandardScaler().fit(Etr)
lin=LogisticRegression(max_iter=1000).fit(scs.transform(Etr),ytr)
lin_acc=accuracy_score(yte,lin.predict(scs.transform(Ete)))
print(f"SimCLR linear-eval test acc={lin_acc:.3f}")

# (2) fine-tune full
def simclr_finetune(train_indices,epochs,freeze=False):
    enc=models.resnet18(weights=None); enc.fc=nn.Identity()
    enc.load_state_dict(simclr.encoder.state_dict())
    m=EncoderClf(enc,freeze=freeze)
    trl=DataLoader(ImgDS(train_indices,tf_aug224),batch_size=64,shuffle=True,num_workers=2)
    train_torch(m,trl,va_full,epochs=epochs,lr=1e-4,log=False)
    a,f,_,_=evaluate(m,te_full); del m,enc; torch.cuda.empty_cache(); return a
ft_full=simclr_finetune(train_idx,epochs=5,freeze=False); print(f"SimCLR fine-tune full acc={ft_full:.3f}")
ft_10=simclr_finetune(kshot(10),epochs=25,freeze=True)
ft_5 =simclr_finetune(kshot(5), epochs=25,freeze=True)
print(f"SimCLR 10-shot={ft_10:.3f}  5-shot={ft_5:.3f}")

In [ ]:
# Q4.2 πίνακας: SimCLR vs ImageNet transfer (αν υπάρχει df31/df32)
rows=[["SimCLR (linear eval)",round(lin_acc,4),"-","-"],
      ["SimCLR (fine-tuned)",round(ft_full,4),round(ft_10,4),round(ft_5,4)]]
if "df31" in globals():
    bi=df31.sort_values("Test Acc",ascending=False).iloc[0]
    rows.append([f"ImageNet transfer ({bi['Model']}/{bi['Strategy']})",bi["Test Acc"],
                 df32.iloc[0]["10-shot"] if "df32" in globals() else "-",
                 df32.iloc[0]["5-shot"] if "df32" in globals() else "-"])
df42=pd.DataFrame(rows,columns=["Method","Full Data","10-shot","5-shot"]); df42

In [ ]:
# Q4.3 t-SNE/UMAP των SimCLR embeddings
vi=np.random.RandomState(SEED).choice(len(Ete),min(VIZ_SUBSET,len(Ete)),replace=False)
emb_t=TSNE(2,random_state=SEED,init="pca",perplexity=30).fit_transform(StandardScaler().fit_transform(Ete[vi]))
emb_u=umap.UMAP(n_components=2,random_state=SEED).fit_transform(StandardScaler().fit_transform(Ete[vi]))
fig,ax=plt.subplots(1,2,figsize=(10,4))
for a,(emb,t) in zip(ax,[(emb_t,"t-SNE"),(emb_u,"UMAP")]):
    for ci,col in zip(range(n_classes),["indianred","steelblue"]):
        mk=yte[vi]==ci; a.scatter(emb[mk,0],emb[mk,1],s=6,alpha=0.5,c=col,label=CLASSES[ci])
    a.set_title(f"SimCLR embeddings — {t}"); a.legend()
plt.tight_layout(); plt.show()

## Task 4.2 — LoRA Fine-Tuning (ViT-B/16)

In [ ]:
# LoRA σε ViT-B/16 (HuggingFace + PEFT). Σύγκριση full fine-tuning vs LoRA.
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model

VIT_NAME="google/vit-base-patch16-224"
class HFWrap(nn.Module):
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,x): return self.m(pixel_values=x).logits

def load_vit():
    return ViTForImageClassification.from_pretrained(VIT_NAME,num_labels=2,ignore_mismatched_sizes=True)

def trainable_stats(m):
    tot=sum(p.numel() for p in m.parameters()); tr=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tr/1e6, 100*tr/tot

results42=[]
# (A) Full fine-tuning
vit=load_vit(); m=HFWrap(vit).to(device)
trm,prc=trainable_stats(vit)
t0=time.time(); train_torch(m,tr_full,va_full,epochs=3,lr=5e-5,log=False); dt=time.time()-t0
acc_full,_,_,_=evaluate(m,te_full)
results42.append(["Full fine-tuning",round(trm,2),round(prc,2),round(acc_full,4),round(dt,1)])
print(f"Full: acc={acc_full:.3f} trainable={trm:.1f}M ({prc:.1f}%) time={dt:.0f}s")
del vit,m; torch.cuda.empty_cache()

# (B) LoRA (auto-detect q/v projection names για transformers 4.x & 5.x)
vit=load_vit()
leaf={n.split('.')[-1] for n,_ in vit.named_modules()}
targets=["q_proj","v_proj"] if "q_proj" in leaf else ["query","value"]
lora=LoraConfig(r=8,lora_alpha=16,target_modules=targets,lora_dropout=0.1,bias="none",modules_to_save=["classifier"])
vit=get_peft_model(vit,lora); m=HFWrap(vit).to(device)
trm,prc=trainable_stats(vit)
t0=time.time(); train_torch(m,tr_full,va_full,epochs=5,lr=1e-4,log=False); dt=time.time()-t0
acc_lora,_,_,_=evaluate(m,te_full)
results42.append([f"LoRA r=8 {targets}",round(trm,2),round(prc,2),round(acc_lora,4),round(dt,1)])
print(f"LoRA: acc={acc_lora:.3f} trainable={trm:.2f}M ({prc:.2f}%) time={dt:.0f}s")
del vit,m; torch.cuda.empty_cache()

df42b=pd.DataFrame(results42,columns=["Method","Trainable(M)","Trainable(%)","Test Acc","Train Time(s)"])
df42b.to_csv(f"{OUT_DIR}/section4_lora.csv",index=False); df42b

**Q4 notes.**
- **Q4.1 augmentations:** Στο contrastive learning οι augmentations ΟΡΙΖΟΥΝ τι θεωρείται «ίδιο». Χωρίς αυτές το NT-Xent έχει trivial λύση. Crop+color jitter είναι τα πιο κρίσιμα (Chen et al. 2020, Table). Στο CIFAKE πρόσεξε: πολύ έντονο color jitter μπορεί να καταστρέψει τα ίδια τα generative artifacts που ξεχωρίζουν real/fake.
- **Q4.4:** Το SSL συνήθως αποδίδει σχετικά μεγαλύτερο όφελος στα low-data regimes (5/10-shot) όπου μια καλή self-supervised αναπαράσταση μετράει πιο πολύ από ετικέτες.
- **Q4.7 LoRA math:** W = W₀ + BA, με B∈ℝ^{d×r}, A∈ℝ^{r×k}, r≪min(d,k). Παγώνεις το W₀ και μαθαίνεις μόνο A,B → trainable params πέφτουν ~2 τάξεις μεγέθους. Δουλεύει γιατί η προσαρμογή σε νέο task έχει χαμηλό intrinsic rank. Πλεονεκτεί όταν: (α) πολλά tasks/adapters πάνω σε ένα frozen backbone (αποθήκευση), (β) περιορισμένη μνήμη/compute για fine-tuning μεγάλων μοντέλων.

# Section 5 — Ενιαίο Benchmark & Συμπεράσματα

## Task 5.1 — Συγκεντρωτικός πίνακας
Μαζεύει αυτόματα τα καλύτερα αποτελέσματα από όλα τα Sections. Όσα δεν έχουν τρέξει εμφανίζονται ως NaN.

In [ ]:
def G(name, d=np.nan): return globals().get(name, d)
def pct(x):
    try: return round(float(x)*100, 2)
    except: return np.nan

rows=[]
# Καλύτερος Κλασικός
if "df1" in globals():
    bc=df1.loc[df1["Test Acc"].idxmax()]
    pdim=feat[bc["Feature"]].shape[1] if bc["Feature"] in feat else np.nan
    rows.append([f"Best Classical ({bc['Algorithm']}+{bc['Feature']})", pct(bc["Test Acc"]), np.nan, np.nan, G("clf_time"), pdim])
# MLP / CNN
rows.append(["MLP", pct(G("mlp_test")), np.nan, np.nan, G("mlp_time"), G("mlp_params")])
rows.append(["Custom CNN", pct(G("cnn_test")), np.nan, np.nan, G("cnn_time"), G("cnn_params")])
# Transfer (best) + few-shot
if "df31" in globals():
    tb=df31.sort_values("Test Acc",ascending=False).iloc[0]; f10=f5=np.nan
    if "df32" in globals():
        mm=df32[df32["Model/Strategy"]==f"{tb['Model']}/{tb['Strategy']}"]
        if len(mm): f10=pct(mm.iloc[0]["10-shot"]); f5=pct(mm.iloc[0]["5-shot"])
    rows.append([f"Transfer ({tb['Model']}/{tb['Strategy']})", pct(tb["Test Acc"]), f10, f5, tb["Train Time(s)"], tb["#Trainable"]])
# Prototype (best of euclid/cosine)
if "df33" in globals():
    p10=pct(df33[df33.Regime=="10-shot"][["Proto-Euclidean","Proto-Cosine"]].max(axis=1).values[0])
    p5 =pct(df33[df33.Regime=="5-shot"][["Proto-Euclidean","Proto-Cosine"]].max(axis=1).values[0])
    rows.append(["Prototype Classifier", np.nan, p10, p5, "~0", np.nan])
# SimCLR
rows.append(["SimCLR (linear eval)", pct(G("lin_acc")), np.nan, np.nan, G("simclr_time"), np.nan])
rows.append(["SimCLR (fine-tuned)", pct(G("ft_full")), pct(G("ft_10")), pct(G("ft_5")), G("simclr_time"), np.nan])
# LoRA
if "df42b" in globals():
    lr=df42b[df42b.Method.str.startswith("LoRA")]
    if len(lr):
        lr=lr.iloc[0]; rows.append(["LoRA (ViT-B/16)", pct(lr["Test Acc"]), np.nan, np.nan, lr["Train Time(s)"], f"{lr['Trainable(M)']}M"])

df5=pd.DataFrame(rows, columns=["Method","Full (%)","10-shot (%)","5-shot (%)","Train Cost (s)","Params"])
df5.to_csv(f"{OUT_DIR}/section5_benchmark.csv", index=False)
df5

In [ ]:
# Grouped bar plot: test accuracy ανά data regime
m=df5["Method"].tolist(); x=np.arange(len(m)); w=0.25
full=df5["Full (%)"].astype(float).values
t10=df5["10-shot (%)"].astype(float).values
t5=df5["5-shot (%)"].astype(float).values
fig,ax=plt.subplots(figsize=(13,5))
ax.bar(x-w, np.nan_to_num(full), w, label="Full Data")
ax.bar(x,   np.nan_to_num(t10),  w, label="10-shot")
ax.bar(x+w, np.nan_to_num(t5),   w, label="5-shot")
ax.set_xticks(x); ax.set_xticklabels(m, rotation=30, ha="right")
ax.set_ylabel("Test accuracy (%)"); ax.set_ylim(0,100)
ax.set_title("CIFAKE — Unified Benchmark (test accuracy ανά data regime)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## Task 5.2 — Insights & Συμπεράσματα

**Q5.1 — Καλύτερη μέθοδος ανά σενάριο.** Διάβασε το `df5`:
- **(α) Πολλά labeled data:** συνήθως κερδίζει το **full fine-tuning** (transfer ή SimCLR fine-tuned) ή το **LoRA** σε ViT — υψηλό acc.
- **(β) 10-shot:** **frozen transfer** ή **prototype classifier** — λιγότερες παράμετροι, λιγότερο overfitting.
- **(γ) 5-shot:** ο **prototype classifier** είναι συχνά ο πιο robust (καθόλου εκπαίδευση βαρών).

**Q5.2 — Κατάταξη υπολογιστικού κόστους (ελαφρύ→βαρύ).** Τυπικά: Prototype (~μόνο embeddings) < Κλασικοί < MLP < Custom CNN < frozen transfer < LoRA < full fine-tuning < SimCLR pretraining. Παράγοντες κόστους: #trainable params, ανάλυση εισόδου (32 vs 224), batch size, epochs, και αν χρειάζεται backward μέσα από όλο το backbone.

**Q5.3 — ImageNet transfer vs self-supervised.** ImageNet transfer: έτοιμες ισχυρές αναπαραστάσεις, γρήγορο, ιδανικό όταν το target μοιάζει με φυσικές εικόνες — αλλά εξαρτάται από labeled ImageNet. SSL (SimCLR): δεν χρειάζεται ετικέτες, μαθαίνει domain-specific features (χρήσιμο για τα ιδιότυπα CIFAKE artifacts) — αλλά απαιτεί ακριβό pretraining και προσεκτικές augmentations.

**Q5.4 — LoRA vs full fine-tuning.** Το LoRA αξίζει όταν το μοντέλο είναι μεγάλο (ViT), η μνήμη/χρόνος περιορισμένα, ή θες πολλά task-specific adapters πάνω σε ένα frozen backbone. Το trade-off accuracy/efficiency γέρνει υπέρ του LoRA όταν η πτώση ακρίβειας είναι μικρή (δες `df42b`) έναντι ~100× λιγότερων trainable params.

**Q5.5 — Future work.** π.χ.: frequency-domain / noise-residual features ειδικά για generative artifacts· δοκιμή σε νεότερα diffusion (SD 2.1/3.0) για generalization· μεγαλύτερα SimCLR batch/epochs· ViT prototype embeddings· ensembles handcrafted+CNN· Grad-CAM για ερμηνεία (όπως το αρχικό CIFAKE paper).